# ANALÍTICA EMPRESARIAL INTEGRADA
## LABORATORIO DIRIGIDO N.° 06

### DISEÑO Y ANÁLISIS DE UN EXPERIMENTO A/B

**Semana 6 · TECSUP 2026-II**  
**Docente:** Pilar Rocío Sayán Mejía  
**Carrera:** Big Data y Ciencia de Datos  
**Plataforma de trabajo:** Google Colab

### Pregunta orientadora

**¿Qué evidencia estadística y práctica necesitamos antes de implementar una variante B?**

> Este notebook acompaña el trabajo desarrollado durante la clase. No es un informe ni un proyecto para exponer. Las celdas alternan explicación breve, ejecución guiada, interpretación y retos de cierre.


# I. Elemento de la capacidad terminal de la semana

Diseña y analiza un experimento A/B identificando el factor, sus niveles, el control, el tratamiento, la unidad experimental y la variable respuesta. Formula las hipótesis estadísticas, establece la métrica primaria, los guardrails, el nivel de significación, la potencia y el efecto mínimo detectable; finalmente, integra la evidencia estadística con la relevancia práctica para sustentar una decisión.


# II. Seguridad y condiciones de trabajo

- Utilice únicamente Google Colab y las fuentes indicadas por la docente.
- No modifique ni fabrique resultados para forzar una conclusión.
- Ejecute las celdas en el orden propuesto y documente cualquier error o dato ausente.
- No comparta claves, credenciales ni información personal en el notebook.
- Mantenga la estación de trabajo ordenada y no instale software no autorizado.


# III. Fundamento teórico
Un A/B test compara un control A con una variante B mediante asignación aleatoria. La unidad experimental de este caso es el usuario, que conserva su variante durante toda la observación. La aleatorización favorece la comparabilidad en expectativa; no garantiza grupos idénticos.

La respuesta es binaria: compra dentro de los siete días posteriores a la asignación. Todos los usuarios completan esa ventana antes del análisis. La métrica primaria es la proporción de usuarios que compran. Se estudia la diferencia de dos proporciones, no ANOVA.

El MDE es el efecto utilizado para planificar la potencia. El umbral empresarial es la mejora que justificaría interés práctico. Aquí son distintos: MDE de 1,5 pp y umbral empresarial de 1,0 pp. La muestra se dimensiona para detectar el MDE frente a cero, no para garantizar evidencia de superar el umbral empresarial.

Se usa una aproximación normal de Wald para la diferencia. El contraste unilateral al 5 % y el límite inferior unilateral del 95 % utilizan el mismo error estándar. También se muestra un IC bilateral del 95 % como descripción de incertidumbre; no se exige que este último excluya cero para reproducir el contraste unilateral.

La recomendación académica requiere evidencia de superar el umbral empresarial y de mantener el guardrail dentro del margen predefinido. Son criterios más exigentes que obtener p < 0,05 frente a cero. Un resultado no concluyente no demuestra igualdad.


# IV. Normas y buenas prácticas empleadas

- Prerregistro lógico: definir hipótesis, métrica, MDE y regla de decisión antes de observar los resultados.
- Aleatorización: evitar que la asignación dependa de características de las unidades.
- Trazabilidad: conservar la fuente, las transformaciones y los parámetros utilizados.
- Reproducibilidad: ejecutar el análisis mediante código verificable.
- No realizar *peeking*: no detener o prolongar el experimento solo porque el resultado momentáneo conviene.
- No contaminar grupos: una unidad debe recibir únicamente la variante asignada.
- Distinguir asociación, evidencia experimental y decisión práctica.


# V. Recursos
- Computadora, Google Colab y notebook LAB-D6-AEI-PSAYAN-2026-02.ipynb.
- Librerías numpy, pandas, scipy, statsmodels y plotly.
- Caso ficticio NexaCommerce con datos simulados generados en el propio notebook. No requiere descargar un dataset externo.

## Naturaleza de los datos utilizados
Los registros son sintéticos y reproducibles mediante una semilla fija. Representan usuarios de un checkout; no proceden de una empresa real. Las probabilidades del simulador son supuestos académicos, no evidencia observada. No cambie la semilla para buscar un resultado favorable. En un estudio real, los datos procederían de registros del experimento y el plan se documentaría antes de recogerlos.


# VI. Metodología para el desarrollo de la sesión

El laboratorio se desarrolla mediante una secuencia guiada de cinco momentos:

1. **Comprender:** recuperar el vocabulario esencial.
2. **Representar:** traducir el caso a la estructura de un diseño experimental.
3. **Formular:** establecer las hipótesis y la regla de decisión antes de observar resultados.
4. **Ejecutar e interpretar:** correr el código por etapas y explicar cada salida.
5. **Decidir y transferir:** integrar evidencia estadística, umbral empresarial y guardrails; luego resolver retos breves.

No se solicita exposición, informe ni trabajo adicional. La evidencia de aprendizaje se construye durante la clase mediante respuestas breves en las pausas de análisis.


# VII. Procedimiento

## Actividad 1 — Activación conceptual

**Propósito:** reconocer los componentes que deben definirse antes de ejecutar un experimento.

Complete la tabla con sus propias palabras. La pregunta de apoyo permite verificar si la definición es suficiente.
DE no debe elegirse después de conocer la diferencia observada.


| **Concepto** | **Definición con mis propias palabras** | **Pregunta de apoyo** |
|---|---|---|
| **Experimento A/B** | Es una prueba en la que se comparan dos versiones, una de control y otra de tratamiento, para evaluar si un cambio produce una diferencia en un resultado. | ¿Qué diferencia existe entre comparar dos grupos y experimentar? |
| **Control** | Es el grupo que mantiene la versión actual y sirve como punto de referencia para comparar los resultados. | ¿Qué representa el punto de referencia? |
| **Tratamiento** | Es el grupo que recibe la nueva versión o cambio que se quiere evaluar. | ¿Qué cambio recibe este grupo? |
| **Unidad experimental** | Es la persona, objeto o unidad sobre la que se aplica una de las variantes del experimento y en la que se observa el resultado. | ¿Sobre quién o qué se aplica la variante? |
| **Variable respuesta** | Es el resultado que se observa en cada unidad experimental después de recibir una de las variantes. | ¿Qué resultado se observa en cada unidad? |
| **Métrica primaria** | Es la medida principal que se utiliza para determinar si el tratamiento logró el resultado esperado. | ¿Cuál será el criterio principal de éxito? |
| **Guardrail** | Es una métrica que se controla para asegurarse de que la mejora de la métrica principal no genere un deterioro en otro resultado importante. | ¿Qué no debería deteriorarse? |
| **MDE** | Es el efecto mínimo que se considera relevante detectar y que se utiliza para planificar la potencia del experimento. | ¿Qué efecto se utiliza para planificar la potencia? |
| **Potencia** | Es la capacidad que tiene el diseño del experimento para detectar el MDE cuando realmente existe. | ¿Qué capacidad tiene el diseño para detectar el MDE? |
| **Peeking** | Es revisar los resultados parciales durante el experimento y tomar una decisión antes de que termine el periodo establecido. | ¿Por qué no se debe decidir mirando resultados parciales? |

**Pausa de verificación**

El MDE no debe elegirse después de conocer la diferencia observada porque el MDE se define antes de ejecutar el experimento y se utiliza para planificar la potencia. Si se eligiera después de observar los resultados, se estaría utilizando la información obtenida del experimento para modificar el efecto que inicialmente se quería detectar.

## Actividad 2 — Interpretación y representación metodológica

### 2.1 Interpretación

**Complete antes de ejecutar:** Determinar si el checkout simplificado de dos pasos aumenta la proporción de usuarios que realizan una compra dentro de siete días, sin superar el margen de deterioro permitido en la incidencia de pago.

### 2.2 Representación del diseño

| Elemento | Definición en este experimento |
|---|---|
| **Factor y niveles** | Checkout: A actual; B simplificado |
| **Unidad experimental** | Usuario único con asignación persistente |
| **Respuesta primaria** | `compra_7d`: 1 si compró dentro de siete días; 0 si no |
| **Métrica primaria** | Usuarios que compraron / todos los usuarios asignados |
| **Guardrail** | `incidencia_pago_7d`: al menos una incidencia por usuario en siete días |
| **Denominador del guardrail** | Todos los usuarios asignados, compren o no |
| **Margen de deterioro** | B−A debe ser inferior a +0,5 pp en incidencias |
| **Baseline histórico supuesto** | Conversión 11,8 %; incidencias 1,8 % |
| **MDE de conversión** | +1,5 pp para planificar potencia de 80 % frente a cero |
| **Umbral empresarial** | Mejora de conversión superior a +1,0 pp |
| **Horizonte** | Reclutamiento planificado de 14 días y 7 días de maduración |
| **Muestra** | Se calcula antes de simular las respuestas |

### 2.3 Hipótesis y reglas previas

**Conversión**

- **H0:** \(p_B-p_A\leq0\)
- **H1:** \(p_B-p_A>0\)
- **α = 0,05**, unilateral.

**Relevancia empresarial:** exigir un límite inferior unilateral del 95 % mayor que **0,010**. Esto evalúa un umbral distinto de cero y no se deduce solo de la estimación puntual.

**Guardrail**

- **H0:** \(q_B-q_A\geq0,005\)
- **H1:** \(q_B-q_A<0,005\)

Se exige que el límite superior unilateral del 95 % sea menor que **0,005**. No encontrar una diferencia significativa no demuestra que el guardrail esté protegido.

La recomendación combina ambos requisitos; se trata de una regla conjunta, no de escoger la prueba favorable. Los intervalos individuales no deben describirse como una región de confianza simultánea.

### Comprobación

**¿Por qué se comparan proporciones? ¿Cómo cambiarían las hipótesis si interesara cualquier diferencia, positiva o negativa?**

**Respuesta:** Se comparan proporciones porque tanto la compra dentro de siete días como la incidencia de pago se expresan como la proporción de usuarios que presentan cada resultado respecto del total de usuarios asignados. Si interesara cualquier diferencia, positiva o negativa, el contraste sería bilateral. En ese caso, la hipótesis nula sería \(H_0:p_B-p_A=0\) y la hipótesis alternativa sería \(H_1:p_B-p_A\neq0\).

## Actividad 3 — Preparación y planificación previa
### Paso 1. Preparar Google Colab
Ejecute las celdas en orden. La configuración y el cálculo de muestra aparecen antes de generar las respuestas. En Colab, la primera celda instala solo los paquetes que faltan. Si se solicita reiniciar, vuelva a ejecutar desde aquí.


In [ ]:
import importlib.util
import subprocess
import sys

paquetes = ["numpy", "pandas", "scipy", "statsmodels", "plotly"]
faltan = [p for p in paquetes if importlib.util.find_spec(p) is None]
if faltan:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
print("Dependencias disponibles.")


Dependencias disponibles.


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from scipy.stats import norm
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

pd.set_option("display.max_columns", 20)
print("Entorno listo.")


Entorno listo.


### Paso 2. Fijar los parámetros antes de observar
Los baselines son supuestos históricos del caso ficticio, no estimaciones obtenidas de los grupos que analizaremos. El margen del guardrail se fija por criterio de negocio. Se calcula muestra para conversión y, por aproximación normal, para no inferioridad del guardrail suponiendo diferencia verdadera cero. Se toma el mayor tamaño. Las potencias se refieren a cada objetivo de diseño; no garantizan 80 % para la regla conjunta ni para superar el umbral empresarial.


In [ ]:
ALPHA = 0.05
POTENCIA = 0.80
BASELINE = 0.118
MDE = 0.015
UMBRAL_NEGOCIO = 0.010
BASE_GUARDRAIL = 0.018
MARGEN_GUARDRAIL = 0.005
SEMILLA = 20260921
DIAS_RECLUTAMIENTO = 14
VENTANA_DIAS = 7

assert 0 < BASELINE < BASELINE + MDE < 1
assert 0 < UMBRAL_NEGOCIO < MDE
analysis = NormalIndPower()
es = proportion_effectsize(BASELINE + MDE, BASELINE)
n_conversion = int(np.ceil(analysis.solve_power(
    effect_size=es, alpha=ALPHA, power=POTENCIA,
    ratio=1, alternative="larger")))
z_alpha = norm.ppf(1 - ALPHA)
z_power = norm.ppf(POTENCIA)
n_guardrail = int(np.ceil(
    2 * BASE_GUARDRAIL * (1 - BASE_GUARDRAIL)
    * (z_alpha + z_power)**2 / MARGEN_GUARDRAIL**2))
N_POR_GRUPO = max(n_conversion, n_guardrail)
print("Muestra por grupo para conversión:", n_conversion)
print("Muestra aproximada por grupo para guardrail:", n_guardrail)
print("Muestra planificada por grupo:", N_POR_GRUPO)
print("Analizar una sola vez al completar muestra y seguimiento.")


Muestra por grupo para conversión: 6027
Muestra aproximada por grupo para guardrail: 8743
Muestra planificada por grupo: 8743
Analizar una sola vez al completar muestra y seguimiento.


### Pausa de análisis 1

**1. ¿Por qué el baseline debe establecerse antes de ver el resultado?**

Porque el baseline es un supuesto histórico que sirve como referencia para diseñar el experimento. Si se estableciera después de observar los resultados, podría verse influenciado por ellos y afectar la planificación de la muestra y la evaluación del experimento.

**2. ¿Por qué la muestra final considera también el guardrail?**

Porque el experimento tiene dos objetivos que deben evaluarse. No solo se busca detectar una mejora en la conversión, sino también comprobar que la incidencia de pago no supere el margen permitido. Por eso se calcula una muestra para cada objetivo y se toma el mayor tamaño. En este caso, conversión requiere **6,027 usuarios por grupo** y el guardrail **8,743**, por lo que se planifican **8,743 usuarios por grupo**.

**3. ¿La potencia calculada garantiza demostrar una mejora superior a 1,0 pp?**

No. La potencia del 80 % se calculó para detectar el **MDE de 1.5 pp**, no para garantizar que se demuestre una mejora superior al umbral empresarial de **1.0 pp**. Además, la potencia calculada corresponde a cada objetivo de diseño y no garantiza una potencia del 80 % para la regla conjunta del experimento.

### Paso 3. Generar y validar el conjunto académico
El simulador asigna aleatoriamente el mismo número de usuarios a A y B. Genera una fila por usuario y completa siete días de observación para todos. Supone conversiones de 11,8 % y 13,8 % e incidencias de 1,8 % en ambos grupos. Son parámetros del generador, no resultados que deban recuperarse exactamente. Las respuestas se simulan independientes para simplificar el ejercicio; una empresa real puede presentar dependencias y estacionalidad no representadas aquí.


In [ ]:
rng = np.random.default_rng(SEMILLA)
total = 2 * N_POR_GRUPO
grupos = rng.permutation(np.repeat(["A_Control", "B_Tratamiento"], N_POR_GRUPO))
prob_compra = np.where(grupos == "A_Control", 0.118, 0.138)
df = pd.DataFrame({
    "usuario_id": np.arange(1, total + 1),
    "grupo": grupos,
    "dia_asignacion": rng.integers(1, DIAS_RECLUTAMIENTO + 1, total),
    "dias_observados": np.full(total, VENTANA_DIAS),
    "compra_7d": rng.binomial(1, prob_compra),
    "incidencia_pago_7d": rng.binomial(1, BASE_GUARDRAIL, total)
})
print("DATOS SIMULADOS — NexaCommerce")
print("Dimensión:", df.shape)
display(df.head())


DATOS SIMULADOS — NexaCommerce
Dimensión: (17486, 6)


,usuario_id,grupo,dia_asignacion,dias_observados,compra_7d,incidencia_pago_7d
0,1,A_Control,8,7,0,0
1,2,B_Tratamiento,5,7,0,0
2,3,A_Control,12,7,0,0
3,4,A_Control,9,7,0,0
4,5,B_Tratamiento,12,7,0,0


In [ ]:
requeridas = {"usuario_id", "grupo", "dias_observados",
              "compra_7d", "incidencia_pago_7d"}
assert requeridas.issubset(df.columns), "Faltan columnas"
assert not df[list(requeridas)].isna().any().any(), "Hay datos ausentes"
assert df["usuario_id"].is_unique, "Usuarios repetidos o contaminados"
assert set(df["grupo"]) == {"A_Control", "B_Tratamiento"}
assert (df["dias_observados"] == VENTANA_DIAS).all()
for col in ["compra_7d", "incidencia_pago_7d"]:
    assert df[col].isin([0, 1]).all(), f"{col} no es binaria"
ab = df.copy()
tam_grupos = ab.groupby("grupo").size()
assert (tam_grupos == N_POR_GRUPO).all()
display(tam_grupos.to_frame("usuarios"))
print("Calidad estructural y seguimiento completos.")


,usuarios
grupo,
A_Control,8743
B_Tratamiento,8743


Calidad estructural y seguimiento completos.


### Pausa de análisis 2
**1. Registre el tamaño de cada grupo. ¿Cumple el plan?**

El grupo **A_Control tiene 8,743 usuarios** y el grupo **B_Tratamiento también tiene 8,743 usuarios**. Sí cumple el plan, porque la muestra definida previamente fue de **8,743 usuarios por grupo**, para un total de **17,486 usuarios**.

**2. ¿Qué ocurriría si un usuario apareciera en ambas variantes?**

Se perdería la asignación exclusiva del experimento, porque un mismo usuario estaría expuesto tanto al control como al tratamiento. Esto podría contaminar la comparación entre A y B y dificultar la interpretación del efecto del tratamiento.

**3. ¿Por qué no debe excluirse a quienes no compraron?**

Porque la métrica principal es la proporción de usuarios que compran entre **todos los usuarios asignados**. Si se excluyera a quienes no compraron, se modificaría el denominador y la conversión podría quedar artificialmente elevada.

**4. ¿Por qué es necesario completar los siete días antes de analizar?**

Porque la variable `compra_7d` mide si el usuario realizó una compra **dentro de los siete días**. Si se analiza antes de completar ese periodo, algunos usuarios todavía podrían comprar y sus resultados estarían incompletos, afectando la comparación entre los grupos.


## Actividad 4 — MDE, potencia y sensibilidad de muestra
### Paso 4. Recuperar el diseño fijado
Revise los parámetros sin recalibrarlos con el resultado observado. El MDE sirve para dimensionar la muestra; el umbral empresarial se utiliza para valorar la mejora. En este caso, las dos cantidades difieren deliberadamente.


In [ ]:
display(pd.DataFrame({
    "parametro": ["Baseline", "MDE", "Umbral empresarial",
                  "Margen guardrail", "Alfa", "Potencia"],
    "valor": [BASELINE, MDE, UMBRAL_NEGOCIO,
              MARGEN_GUARDRAIL, ALPHA, POTENCIA]
}))
print("Muestra final por grupo:", N_POR_GRUPO)


,parametro,valor
0,Baseline,0.118
1,MDE,0.015
2,Umbral empresarial,0.010
3,Margen guardrail,0.005
4,Alfa,0.050
5,Potencia,0.800


Muestra final por grupo: 8743


### Paso 5. Explorar la sensibilidad al MDE
Compare MDE de 0,5, 1,5 y 2,5 pp. Esta exploración es didáctica y no cambia el plan ni la decisión del caso ya fijado.
**Predicción:** ¿qué efecto necesitará más muestra y por qué?


In [ ]:
filas = []
for mde in [0.005, 0.015, 0.025]:
    es = proportion_effectsize(BASELINE + mde, BASELINE)
    n = analysis.solve_power(effect_size=es, alpha=ALPHA,
                             power=POTENCIA, alternative="larger")
    filas.append({"MDE_pp": mde * 100,
                  "n_conversion": int(np.ceil(n)),
                  "n_con_guardrail": max(int(np.ceil(n)), n_guardrail)})
sensibilidad_n = pd.DataFrame(filas)
display(sensibilidad_n)
fig = px.line(sensibilidad_n, x="MDE_pp",
              y=["n_conversion", "n_con_guardrail"], markers=True,
              title="Muestra por grupo según el MDE",
              labels={"value": "Usuarios por grupo", "MDE_pp": "MDE (pp)"})
fig.show()


,MDE_pp,n_conversion,n_con_guardrail
0,0.5,52413,52413
1,1.5,6027,8743
2,2.5,2241,8743


### Pausa de análisis 3

**1. Explique por qué reducir el MDE aumenta la muestra para conversión.**

Porque un MDE más pequeño representa una mejora más difícil de detectar. Por eso se necesitan más usuarios para poder identificar ese efecto manteniendo el alfa y la potencia establecidos. En este caso, detectar **0.5 pp** requiere **52,413 usuarios por grupo**, mientras que detectar **2.5 pp** requiere **2,241**.

**2. ¿Por qué el guardrail puede determinar la muestra final?**

Porque la muestra debe ser suficiente para cumplir tanto el objetivo principal como la evaluación del guardrail. Si el guardrail requiere una muestra mayor que la conversión, se toma ese tamaño como muestra final. En este caso, el guardrail requiere **8,743 usuarios por grupo**, frente a **6,027** para conversión.

**3. ¿Sería válido aumentar el MDE después de ver los resultados para declarar éxito?**

No. El MDE debe fijarse antes de observar los resultados porque forma parte del diseño y dimensionamiento de la muestra. Cambiarlo después de ver los resultados modificaría el criterio definido previamente y podría favorecer una conclusión de éxito.

## Actividad 5 — Análisis estadístico de la comparación A/B

**Propósito:** calcular el efecto observado y evaluar la hipótesis mediante una prueba para dos proporciones.

### Paso 6. Obtener tasas y tamaño del efecto observado

Antes de ejecutar, recuerde:

- Diferencia absoluta: $p_B-p_A$.
- Lift relativo: $(p_B-p_A)/p_A$.
- La diferencia se expresa en puntos porcentuales; el lift expresa un cambio relativo respecto de A.


In [ ]:
tabla = ab.groupby("grupo")["compra_7d"].agg(["sum", "count", "mean"])
tabla.columns = ["compras", "n", "tasa_conversion"]
x_A, n_A = int(tabla.loc["A_Control", "compras"]), int(tabla.loc["A_Control", "n"])
x_B, n_B = int(tabla.loc["B_Tratamiento", "compras"]), int(tabla.loc["B_Tratamiento", "n"])
p_A, p_B = x_A / n_A, x_B / n_B
efecto = p_B - p_A
lift_rel = efecto / p_A if p_A > 0 else np.nan
display(tabla)
print(f"Diferencia B−A: {efecto*100:.3f} pp")
print(f"Lift relativo: {lift_rel:.2%}")


,compras,n,tasa_conversion
grupo,,,
A_Control,996,8743,0.11392
B_Tratamiento,1236,8743,0.14137


Diferencia B−A: 2.745 pp
Lift relativo: 24.10%


### Pausa de análisis 4

Complete sin utilizar todavía el p-valor:

> La tasa de B es **14.137 %** y la tasa de A es **11.392 %**. La diferencia observada es **2.745 puntos porcentuales**, equivalente a un lift relativo de **24.10 %**. Estos valores describen la muestra, pero todavía no permiten afirmar que **la mejora observada en B sea estadísticamente significativa ni que se haya demostrado el efecto planteado en la hipótesis**.

### Paso 7. Contraste e intervalos coherentes
Usamos la aproximación normal de Wald con error estándar no combinado. Se comprueba que haya al menos diez compras y diez no compras en cada grupo; esta regla práctica no sustituye la revisión del diseño.

Para la prueba unilateral frente a cero, p < 0,05 equivale a límite inferior unilateral del 95 % > 0 con el mismo método. El IC bilateral del 95 % se presenta aparte. La función también permite contrastar un umbral distinto de cero. El p-valor no es la probabilidad de que H0 sea verdadera.


In [ ]:
def diferencia_wald(x_b, n_b, x_a, n_a, umbral=0.0):
    if min(x_b, n_b-x_b, x_a, n_a-x_a) < 10:
        raise ValueError("Frecuencias pequeñas: revisar método antes de continuar")
    pb, pa = x_b/n_b, x_a/n_a
    d = pb-pa
    se = np.sqrt(pb*(1-pb)/n_b + pa*(1-pa)/n_a)
    z = (d-umbral)/se
    return {"d": d, "se": se, "z": z,
            "p_mayor": norm.sf(z), "p_menor": norm.cdf(z),
            "li_uni95": d-norm.ppf(1-ALPHA)*se,
            "ls_uni95": d+norm.ppf(1-ALPHA)*se,
            "ic95_low": d-norm.ppf(1-ALPHA/2)*se,
            "ic95_high": d+norm.ppf(1-ALPHA/2)*se}

conv = diferencia_wald(x_B, n_B, x_A, n_A)
negocio = diferencia_wald(x_B, n_B, x_A, n_A, UMBRAL_NEGOCIO)
print(f"z frente a cero: {conv['z']:.4f}")
print(f"p unilateral frente a cero: {conv['p_mayor']:.6g}")
print(f"Límite inferior unilateral 95%: {conv['li_uni95']*100:.3f} pp")
print(f"IC bilateral 95%: [{conv['ic95_low']*100:.3f}, {conv['ic95_high']*100:.3f}] pp")
print(f"p unilateral frente a +1,0 pp: {negocio['p_mayor']:.6g}")


z frente a cero: 5.4436
p unilateral frente a cero: 2.61091e-08
Límite inferior unilateral 95%: 1.916 pp
IC bilateral 95%: [1.757, 3.733] pp
p unilateral frente a +1,0 pp: 0.000269553


In [ ]:
grafico = tabla.reset_index()
grafico["conversion_pct"] = grafico["tasa_conversion"] * 100
fig = px.bar(grafico, x="grupo", y="conversion_pct", text="conversion_pct",
             title="Conversión observada — datos simulados",
             labels={"conversion_pct": "Usuarios que compraron (%)", "grupo": "Grupo"})
fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.update_yaxes(rangemode="tozero")
fig.show()


### Análisis y conclusión estadística

**1. Registre alfa, z y p unilateral frente a cero.**

El nivel de significancia es **α = 0.05**, el estadístico es **z = 5.4436** y el p-valor unilateral frente a cero es **2.61091 × 10⁻⁸**.

**2. ¿Se rechaza H0? Explique en el contexto de conversión del caso simulado.**

Sí, se rechaza **H0**, porque el p-valor es menor que α = 0.05. En el caso simulado, existe evidencia estadística para concluir que la proporción de usuarios que compran en 7 días es mayor en el grupo B que en el grupo A.

**3. ¿El límite inferior unilateral supera cero? ¿Y el umbral de 1,0 pp?**

Sí. El límite inferior unilateral del 95 % es **+1.916 pp**, por lo que supera cero. Además, **1.916 pp también supera el umbral empresarial de +1.0 pp**. El contraste directo frente a +1.0 pp también resulta significativo, con **p = 0.000269553**.

**4. Explique por qué el IC bilateral del 95 % y el límite unilateral del 95 % tienen extremos distintos.**

Porque responden a contrastes diferentes. El **IC bilateral del 95 %** reserva el 5 % de error entre los dos extremos, mientras que el **límite unilateral del 95 %** concentra ese 5 % en un solo extremo. Por eso el intervalo bilateral obtenido es **[1.757, 3.733] pp**, mientras que el límite inferior unilateral es **+1.916 pp**.

**5. ¿Superar 1,0 pp con la estimación puntual demuestra superar ese umbral con evidencia suficiente?**

No. Que la estimación puntual sea de **+2.745 pp** y supere +1.0 pp no es suficiente por sí solo. También se debe evaluar estadísticamente si existe evidencia de que la diferencia verdadera supera el umbral de +1.0 pp. En este caso, el contraste frente a +1.0 pp da **p = 0.000269553**, por lo que sí existe evidencia estadística para superar ese umbral bajo el método utilizado.


## Actividad 6 — Del resultado estadístico a la decisión
### Paso 8. Evaluar el guardrail con incertidumbre
El guardrail es la proporción de usuarios con alguna incidencia de pago en siete días. El margen de deterioro admisible es +0,5 pp. La no inferioridad exige que el límite superior unilateral del 95 % esté por debajo de ese margen. No basta obtener p > 0,05 en una prueba de igualdad.


In [ ]:
guardrail = ab.groupby("grupo")["incidencia_pago_7d"].agg(["sum", "count", "mean"])
guardrail.columns = ["incidencias", "n", "tasa_incidencia"]
g_A = int(guardrail.loc["A_Control", "incidencias"])
g_B = int(guardrail.loc["B_Tratamiento", "incidencias"])
guard = diferencia_wald(g_B, n_B, g_A, n_A, MARGEN_GUARDRAIL)
display(guardrail)
print(f"Cambio del guardrail B−A: {guard['d']*100:.3f} pp")
print(f"Límite superior unilateral 95%: {guard['ls_uni95']*100:.3f} pp")
print(f"Margen permitido: {MARGEN_GUARDRAIL*100:.1f} pp")
print(f"p de no inferioridad: {guard['p_menor']:.6g}")


,incidencias,n,tasa_incidencia
grupo,,,
A_Control,139,8743,0.015898
B_Tratamiento,176,8743,0.020130


Cambio del guardrail B−A: 0.423 pp
Límite superior unilateral 95%: 0.754 pp
Margen permitido: 0.5 pp
p de no inferioridad: 0.351286


### Paso 9. Integrar la regla de decisión
| Criterio | Evidencia requerida |
|---|---|
| Diseño y calidad | Una fila por usuario, asignación persistente, muestra y seguimiento completos |
| Mejora frente a cero | p unilateral < 0,05 |
| Relevancia empresarial | Límite inferior unilateral del 95 % > +1,0 pp |
| Guardrail | Límite superior unilateral del 95 % < +0,5 pp |

La regla se fijó antes de simular. El MDE dimensiona la muestra y no reemplaza al umbral empresarial. Si no se satisface un requisito, distinguir evidencia insuficiente de evidencia de perjuicio. Cualquier experimento posterior necesita un nuevo plan; no seguir agregando datos hasta lograr significancia.


In [ ]:
cumple_estadistica = bool(conv["p_mayor"] < ALPHA)
cumple_negocio = bool(conv["li_uni95"] > UMBRAL_NEGOCIO)
cumple_guardrail = bool(guard["ls_uni95"] < MARGEN_GUARDRAIL)
cumple_diseno = bool((tam_grupos >= N_POR_GRUPO).all())
criterios = pd.Series({"Diseño validado": cumple_diseno,
    "Mejora frente a cero": cumple_estadistica,
    "Mejora superior al umbral": cumple_negocio,
    "Guardrail protegido": cumple_guardrail})
display(criterios.to_frame("cumple"))
if criterios.all():
    decision = "Recomendar B en el caso simulado, con seguimiento posterior."
else:
    decision = "No recomendar el despliegue todavía: revisar los criterios no cumplidos."
print(decision)
print("Conclusión académica con datos simulados; no valida un despliegue real.")


,cumple
Diseño validado,True
Mejora frente a cero,True
Mejora superior al umbral,True
Guardrail protegido,False


No recomendar el despliegue todavía: revisar los criterios no cumplidos.
Conclusión académica con datos simulados; no valida un despliegue real.


### Pausa de decisión

| Componente | Resultado observado | Interpretación |
|---|---|---|
| p unilateral frente a cero | **2.61091 × 10⁻⁸** | Es menor que α = 0.05, por lo que existe evidencia de una mejora de B frente a A en conversión. |
| Diferencia B−A en pp y lift relativo | **+2.745 pp** y **+24.10 %** | B presenta una mayor conversión observada que A. |
| Límite inferior y umbral empresarial | **+1.916 pp > +1.0 pp** | El límite inferior unilateral supera el umbral empresarial, por lo que se cumple este criterio. |
| Límite superior del guardrail y margen | **+0.754 pp > +0.5 pp** | No se demuestra que el deterioro del guardrail esté dentro del margen permitido. |
| Calidad y límites del caso simulado | **8,743 usuarios por grupo; 7 días de observación** | El diseño y seguimiento están completos, pero los datos son simulados y no representan un despliegue real. |

**Decisión en cuatro líneas:**

**No recomendar todavía**, porque aunque B mejora la conversión en **+2.745 pp** y el límite inferior es **+1.916 pp**, superior al umbral empresarial de +1.0 pp.  
El guardrail no cumple, ya que su límite superior unilateral es **+0.754 pp**, mayor que el margen permitido de +0.5 pp.  
Por ello, no se cumplen todos los criterios de la regla conjunta.  
Se debe plantear un **nuevo diseño experimental**, revisando el tratamiento y el riesgo del guardrail, y calcular nuevamente la muestra y los criterios antes de recolectar nuevos datos; no continuar esta muestra buscando significancia.

## Actividad 7 — Retos de aplicación y retroalimentación
Se resuelven durante la clase y no generan una entrega adicional.

### Reto 1 — Estimación y umbral

La prueba con **p = 0.02** frente a cero proporciona evidencia estadística de una diferencia respecto de cero, porque el p-valor es menor que 0.05. Sin embargo, una mejora estimada de **0.4 pp** no demuestra que se supere el umbral empresarial de **3 pp**. El intervalo aporta información sobre la incertidumbre de la estimación y permite evaluar si los valores compatibles con los datos alcanzan o superan el umbral de negocio.

### Reto 2 — Cambio de pregunta

Si interesa cualquier diferencia entre A y B, las hipótesis serían:

**H0:** pB − pA = 0  
**H1:** pB − pA ≠ 0

Se utilizaría un **contraste bilateral**, porque interesa detectar tanto una diferencia positiva como una negativa. Debe elegirse antes de observar los resultados para mantener definido el criterio de análisis y evitar cambiar la pregunta después de conocer los datos.

### Reto 3 — Guardrail no concluyente

No. Con un aumento observado de **+0.2 pp**, el resultado puntual está dentro del margen de **+0.5 pp**, pero el límite superior unilateral llega a **+0.7 pp**, superando el margen permitido. Por eso no se ha demostrado que el guardrail cumpla. Tampoco es correcto afirmar que se ha demostrado perjuicio, porque el resultado no demuestra que el deterioro supere necesariamente el margen; existe incertidumbre y la evidencia es insuficiente para concluir el cumplimiento del guardrail.

### Reto 4 — Reproducibilidad y sesgo

Cambiar la semilla hasta obtener una decisión favorable invalida el caso base porque introduce una selección de resultados después de observar las simulaciones. La semilla debe fijarse previamente y mantenerse para que el resultado sea reproducible. Si se quieren explorar otros escenarios, se pueden documentar por separado, indicando las semillas, parámetros modificados y resultados obtenidos, sin reemplazar ni presentar esa exploración como el caso base.

# VIII. Secuencia de trabajo para la sesión
| Momento | Minutos | Desarrollo |
|---|---:|---|
| Apertura | 10 | Pregunta orientadora y caso |
| Actividad 1 | 15 | Activación conceptual |
| Actividad 2 | 20 | Representación e hipótesis |
| Actividad 3 | 30 | Planificación, simulación y calidad |
| Receso | 15 | Pausa |
| Actividad 4 | 25 | Sensibilidad de muestra |
| Actividad 5 | 30 | Tasas, contraste e intervalos |
| Actividad 6 | 20 | Guardrail y decisión |
| Actividad 7 | 15 | Retos y ticket de salida |

Total: 180 minutos, incluido el receso.

## Ticket de salida

**«Un experimento A/B no termina en el p-valor porque…»**

Un experimento A/B no termina en el p-valor porque **también se debe evaluar el tamaño del efecto, la relevancia empresarial, la incertidumbre mediante intervalos y los guardrails definidos previamente**. En este caso, aunque la conversión de B mostró una mejora estadísticamente significativa, el guardrail no cumplió el margen permitido. Por eso, **el p-valor por sí solo no determina la decisión final**.


# IX. Lista de verificación de aprendizaje
- [X] Identifico factor, niveles y unidad experimental.
- [X] Defino respuesta, denominador y ventana de observación.
- [X] Formulo hipótesis antes de observar resultados.
- [X] Distingo baseline, MDE, umbral empresarial, alfa y potencia.
- [X] Calculo muestra antes de generar los datos del caso.
- [X] Interpreto pp, lift, p-valor e intervalos sin confundirlos.
- [X] Evalúo el guardrail con margen e incertidumbre.
- [X] Distingo evidencia insuficiente de evidencia de perjuicio.
- [X] Reconozco peeking, contaminación y selección de resultados.
- [X] Declaro el carácter simulado y los límites del ejercicio.


# X. Referencias
- Kohavi, R., Tang, D., y Xu, Y. (2020). Trustworthy Online Controlled Experiments. Cambridge University Press.
- Bruce, P., Bruce, A., y Gedeck, P. (2020). Practical Statistics for Data Scientists, segunda edición. O’Reilly.
- statsmodels. Documentación de NormalIndPower y proportion_effectsize: https://www.statsmodels.org/stable/stats.html
- SciPy. Distribución normal, funciones cdf, sf y ppf: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html

Datos: simulación académica NexaCommerce incluida en el notebook, semilla 20260921. Los datos no proceden de las referencias bibliográficas.
